
# Milestone 06 — Model Improvement & Class Imbalance Handling

## Project: Bank Account Fraud Detection — Deep Learning & MLOps

### Objectives

Improve our Milestone 05 PyTorch MLP through controlled experiments while maintaining reproducibility and preventing data leakage.

### Starting Benchmarks

| Model | Validation PR-AUC |
|---|---:|
| DummyClassifier | 0.013405 |
| Logistic Regression | 0.154819 |
| PyTorch MLP — Epoch 5 | 0.157469 |
| HistGradientBoosting | 0.166961 |

**Primary evaluation metric:** PR-AUC (Average Precision)

### Experiment Phases

- **06A:** Experiment tracking and configuration management
- **06B:** Google Colab integration and CPU/GPU benchmarking
- **06C:** Class imbalance strategies and hyperparameter tuning
- **06D:** Model selection, reproducibility checks, and reporting

### Data Leakage Rules

- Training: Months 0–5
- Validation: Month 6
- Test: Month 7 — SEALED
- Fit preprocessing on training data only.
- Do not access the test set during model development.
- Apply any sampling or class-balancing procedures to training data only.

### Experiment Standards

Each experiment must record:

- Experiment identifier and random seed
- Model architecture and hyperparameters
- Loss function, optimizer, and learning rate
- Class imbalance strategy
- Training and validation metrics
- Training device and execution duration
- Model selection decision and justification

### Success Criteria

1. Establish a reliable experiment-tracking workflow.
2. Compare training performance on local CPU and Google Colab GPU.
3. Evaluate multiple controlled model improvements.
4. Compare results against the fixed Milestone 05 baseline.
5. Preserve reproducibility and maintain the sealed test set.


### Environment & Reproducibility Setup

In [ ]:

# ============================================================
# Milestone 06 — Cell 2: Environment & Reproducibility
# ============================================================

from pathlib import Path
import sys
import random
import platform

import numpy as np
import pandas as pd
import torch
import sklearn

# 1. Locate our project directory
PROJECT_ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "src/models/mlp.py").is_file()
        and (path / "notebooks/05_pytorch_mlp.ipynb").is_file()
    ),
    None
)

assert PROJECT_ROOT is not None, "Project root not found"

# 2. Make our reusable modules importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 3. Configure reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 4. Detect available hardware
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 5. Display environment information
print("PROJECT ENVIRONMENT")
print("=" * 55)

print("Project root:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nREPRODUCIBILITY")
print("=" * 55)

print("Random seed:", SEED)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nEnvironment verification: PASS")


### Experiment Configuration

In [ ]:

# ============================================================
# Milestone 06 — Cell 3: Experiment Configuration
# ============================================================

from dataclasses import dataclass, asdict
import json

@dataclass(frozen=True)
class ExperimentConfig:
    experiment_id: str = "m06_reference_mlp_config"

    # Reproducibility
    seed: int = 42
    device: str = str(DEVICE)

    # Model architecture
    input_features: int = 55
    hidden_layers: tuple[int, int] = (64, 32)
    dropout_rate: float = 0.10

    # Training hyperparameters
    epochs: int = 5
    batch_size: int = 1024
    learning_rate: float = 0.001
    optimizer: str = "Adam"
    loss_function: str = "BCEWithLogitsLoss"

    # Class imbalance strategy
    imbalance_strategy: str = "none"

    # Temporal data boundaries
    train_months: tuple[int, ...] = (0, 1, 2, 3, 4, 5)
    validation_months: tuple[int, ...] = (6,)
    test_month: int = 7
    test_status: str = "SEALED"


# Create the reference configuration
config = ExperimentConfig()

# Validate configuration
assert config.input_features == 55
assert config.epochs > 0
assert config.batch_size > 0
assert config.learning_rate > 0
assert 0 <= config.dropout_rate < 1

# Prevent accidental overlap between dataset splits
assert set(config.train_months).isdisjoint(
    config.validation_months
)
assert config.test_month not in (
    *config.train_months,
    *config.validation_months,
)
assert config.test_status == "SEALED"

# Display configuration as readable JSON
print("EXPERIMENT CONFIGURATION")
print("=" * 55)
print(json.dumps(asdict(config), indent=2))

print("\nConfiguration validation: PASS")
print("Temporal split protection: PASS")
print("Experiment tracking setup: PASS")


### Build Our Experiment Registry

In [ ]:

# ============================================================
# Milestone 06 — Cell 4: Experiment Registry
# ============================================================

# 1. Locate our verified Milestone 05 metadata
BASELINE_METADATA_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "milestone_05_reference"
    / "milestone_05_mlp_metadata.json"
)

assert BASELINE_METADATA_PATH.is_file(), (
    f"Baseline metadata not found: {BASELINE_METADATA_PATH}"
)

# 2. Load historical experiment information
with open(BASELINE_METADATA_PATH, "r", encoding="utf-8") as file:
    baseline_metadata = json.load(file)

architecture = baseline_metadata["architecture"]
training = baseline_metadata["training"]
results = baseline_metadata["validation_results"]

# 3. Confirm compatibility with our reference configuration
assert architecture["input_features"] == config.input_features
assert tuple(architecture["hidden_layers"]) == config.hidden_layers
assert architecture["dropout"] == config.dropout_rate
assert training["epochs"] == config.epochs
assert training["random_seed"] == config.seed
assert baseline_metadata["data"]["test_status"] == "SEALED"

# 4. Create our first registry record
baseline_record = {
    "experiment_id": "m05_fixed_budget_baseline",
    "source": "milestone_05_reference",
    "status": "completed_imported",
    "device": training["device"],
    "seed": training["random_seed"],
    "epochs": training["epochs"],
    "learning_rate": training["learning_rate"],
    "dropout_rate": architecture["dropout"],
    "imbalance_strategy": "none",
    "validation_pr_auc": float(results["pr_auc"]),
    "validation_roc_auc": float(results["roc_auc"]),
    "validation_loss": float(results["validation_loss"]),
}

# 5. Initialize registry
experiment_registry = [baseline_record]

# Convert to a DataFrame for convenient comparisons
registry_df = pd.DataFrame(experiment_registry)

# 6. Verify the registry
assert len(registry_df) == 1
assert registry_df["experiment_id"].is_unique
assert registry_df["validation_pr_auc"].between(0, 1).all()
assert registry_df["status"].iloc[0] == "completed_imported"

# 7. Display the initial registry
print("MILESTONE 06 — EXPERIMENT REGISTRY")
print("=" * 65)

display(
    registry_df[
        [
            "experiment_id",
            "device",
            "imbalance_strategy",
            "epochs",
            "validation_pr_auc",
            "validation_roc_auc",
            "status",
        ]
    ].round(6)
)

print("\nHistorical baseline imported: PASS")
print("Experiment registry initialized: PASS")
print("Month-7 test set remains: SEALED")


### Build a Safe Experiment Registration Function

In [ ]:

# ============================================================
# Milestone 06 — Cell 5: Safe Experiment Registration
# ============================================================

REQUIRED_EXPERIMENT_FIELDS = {
    "experiment_id",
    "source",
    "status",
    "device",
    "seed",
    "epochs",
    "learning_rate",
    "dropout_rate",
    "imbalance_strategy",
    "validation_pr_auc",
    "validation_roc_auc",
    "validation_loss",
}


def register_experiment(registry: list, record: dict) -> None:
    """Validate and register one completed experiment."""

    # 1. Require all expected fields
    missing = REQUIRED_EXPERIMENT_FIELDS - record.keys()

    if missing:
        raise ValueError(
            f"Missing experiment fields: {sorted(missing)}"
        )

    # 2. Prevent accidental test metric registration
    if any(str(key).startswith("test_") for key in record):
        raise ValueError(
            "Test-set metrics are prohibited during Milestone 06."
        )

    # 3. Require unique experiment identifiers
    existing_ids = {r["experiment_id"] for r in registry}

    if record["experiment_id"] in existing_ids:
        raise ValueError(
            f"Duplicate experiment ID: {record['experiment_id']}"
        )

    # 4. Validate numerical metrics
    for metric in (
        "validation_pr_auc",
        "validation_roc_auc",
        "validation_loss",
    ):
        value = float(record[metric])

        if not np.isfinite(value):
            raise ValueError(f"{metric} must be finite")

        if metric in (
            "validation_pr_auc",
            "validation_roc_auc",
        ):
            if not 0.0 <= value <= 1.0:
                raise ValueError(f"{metric} must be between 0 and 1")

        if metric == "validation_loss" and value < 0:
            raise ValueError("Validation loss cannot be negative")

    # 5. Register only after all checks pass
    registry.append(record.copy())


# ------------------------------------------------------------
# Verify the registry safeguards without adding a fake run
# ------------------------------------------------------------

initial_count = len(experiment_registry)

# Test duplicate prevention using our imported baseline
try:
    register_experiment(experiment_registry, baseline_record)

except ValueError as error:
    assert "Duplicate experiment ID" in str(error)
    print("Duplicate experiment protection: PASS")

else:
    raise AssertionError("Duplicate experiment was accepted")


# Confirm the registry remains unchanged
assert len(experiment_registry) == initial_count == 1

# Confirm no test metrics appear in the registry
assert all(
    not any(str(key).startswith("test_") for key in record)
    for record in experiment_registry
)

print("Registry integrity: PASS")
print("Test metric exclusion: PASS")
print("Safe experiment registration: PASS")


### Experiment Comparison & Portable Results

In [ ]:

# ============================================================
# Milestone 06 — Cell 6: Experiment Comparison & Portable Results
# ============================================================

# 1. Retrieve verified historical benchmarks
BASELINE_EXPERIMENT_ID = "m05_fixed_budget_baseline"

baseline_pr_auc = float(
    baseline_record["validation_pr_auc"]
)

hgb_pr_auc = float(
    results["histgradientboosting_pr_auc"]
)

# 2. Rebuild the table from our current registry
comparison_df = pd.DataFrame(experiment_registry).copy()

assert comparison_df["experiment_id"].is_unique
assert BASELINE_EXPERIMENT_ID in comparison_df["experiment_id"].values

# 3. Calculate improvement over the MLP baseline
comparison_df["delta_vs_m05"] = (
    comparison_df["validation_pr_auc"] - baseline_pr_auc
)

# 4. Calculate how far each model is from the HGB benchmark
comparison_df["gap_to_hgb"] = (
    hgb_pr_auc - comparison_df["validation_pr_auc"]
)

# 5. Identify experiments that outperform HGB
comparison_df["beats_hgb"] = (
    comparison_df["validation_pr_auc"] > hgb_pr_auc
)

# 6. Sort experiments by our primary evaluation metric
comparison_df = comparison_df.sort_values(
    by="validation_pr_auc",
    ascending=False
).reset_index(drop=True)

# 7. Create a portable experiment snapshot
portable_registry = {
    "schema_version": 1,
    "milestone": "06",
    "primary_metric": "validation_pr_auc",
    "baseline_experiment_id": BASELINE_EXPERIMENT_ID,
    "baseline_pr_auc": baseline_pr_auc,
    "hgb_benchmark_pr_auc": hgb_pr_auc,
    "holdout_status": "SEALED",
    "reference_config": asdict(config),
    "experiments": [record.copy() for record in experiment_registry],
}

# Ensure the entire snapshot can be serialized as JSON
portable_registry_json = json.dumps(
    portable_registry,
    indent=2,
    allow_nan=False
)

# 8. Verify calculated comparisons
baseline_row = comparison_df.loc[
    comparison_df["experiment_id"] == BASELINE_EXPERIMENT_ID
].iloc[0]

assert np.isclose(baseline_row["delta_vs_m05"], 0.0)
assert baseline_row["gap_to_hgb"] > 0
assert portable_registry["holdout_status"] == "SEALED"
assert len(portable_registry["experiments"]) == 1

# 9. Display the results
print("MILESTONE 06 — EXPERIMENT COMPARISON")
print("=" * 65)

display(
    comparison_df[
        [
            "experiment_id",
            "validation_pr_auc",
            "delta_vs_m05",
            "gap_to_hgb",
            "beats_hgb",
        ]
    ].round(6)
)

print("\nEXPERIMENT TRACKING")
print("Registered experiments:", len(experiment_registry))
print("MLP reference PR-AUC:", round(baseline_pr_auc, 6))
print("HGB benchmark PR-AUC:", round(hgb_pr_auc, 6))

print("\nComparison calculations: PASS")
print("Portable JSON serialization: PASS")
print("Holdout status: SEALED")
print("Experiment comparison setup: PASS")


### Safely Export the Experiment Registry

In [ ]:

# ============================================================
# Milestone 06 — Cell 7: Safe Experiment Registry Export
# ============================================================

import hashlib

# 1. Define the destination
EXPORT_DIR = PROJECT_ROOT / "artifacts" / "milestone_06"

REGISTRY_EXPORT_PATH = (
    EXPORT_DIR / "experiment_registry_v01.json"
)

# 2. Verify our in-memory experiment registry
assert len(experiment_registry) == 1
assert portable_registry["holdout_status"] == "SEALED"
assert portable_registry["primary_metric"] == "validation_pr_auc"

assert all(
    not any(str(key).startswith("test_") for key in record)
    for record in portable_registry["experiments"]
)

# 3. Prevent accidental overwriting
if REGISTRY_EXPORT_PATH.exists():
    raise FileExistsError(
        f"Experiment registry already exists: {REGISTRY_EXPORT_PATH}. "
        "Use a new versioned filename instead of overwriting."
    )

# 4. Create the output directory
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 5. Save JSON with exclusive file creation
export_text = portable_registry_json + "\n"

with REGISTRY_EXPORT_PATH.open(
    "x", encoding="utf-8", newline="\n"
) as file:
    file.write(export_text)

# 6. Reload the exported JSON
with REGISTRY_EXPORT_PATH.open(
    "r", encoding="utf-8"
) as file:
    reloaded_registry = json.load(file)

# 7. Validate saved content against the original snapshot
expected_registry = json.loads(portable_registry_json)

assert reloaded_registry == expected_registry
assert reloaded_registry["holdout_status"] == "SEALED"
assert len(reloaded_registry["experiments"]) == 1

# 8. Verify byte-level integrity
saved_bytes = REGISTRY_EXPORT_PATH.read_bytes()
expected_bytes = export_text.encode("utf-8")

assert saved_bytes == expected_bytes

registry_sha256 = hashlib.sha256(saved_bytes).hexdigest()

# 9. Display verification results
print("MILESTONE 06 — EXPERIMENT REGISTRY EXPORT")
print("=" * 60)

print("File:", REGISTRY_EXPORT_PATH.name)
print("Location:", REGISTRY_EXPORT_PATH)
print("File exists:", REGISTRY_EXPORT_PATH.is_file())
print("Registered experiments:", len(reloaded_registry["experiments"]))
print("File size:", len(saved_bytes), "bytes")
print("SHA-256:", registry_sha256)

print("\nRegistry JSON reload: PASS")
print("File integrity verification: PASS")
print("Overwrite protection: ENABLED")
print("Holdout status: SEALED")
print("Experiment registry export: PASS")


### Reconstruct Our 55-Feature Development Arrays

In [ ]:

# ============================================================
# Milestone 06 — Cell 8: Prepare Colab Development Arrays
# ============================================================

import joblib
import json

from src.data import splitting
from src.features import preprocessing

# 1. Reuse existing project configurations
raw_path = PROJECT_ROOT / "data/raw/Base.csv"
split_path = PROJECT_ROOT / "configs/split_config.json"
prep_path = PROJECT_ROOT / "configs/preprocessing_config.json"

# Trusted, locally generated Milestone 05 artifacts
preprocessor_path = (
    PROJECT_ROOT / "models/milestone_05_preprocessor.joblib"
)

metadata_path = (
    PROJECT_ROOT
    / "artifacts/milestone_05_reference"
    / "milestone_05_mlp_metadata.json"
)

for path in (
    raw_path,
    split_path,
    prep_path,
    preprocessor_path,
    metadata_path,
):
    assert path.is_file(), f"Missing file: {path}"

split_cfg = splitting.load_split_config(split_path)
prep_cfg = preprocessing.load_preprocessing_config(prep_path)

# 2. Verify temporal boundaries
assert split_cfg["train"]["months"] == list(range(6))
assert split_cfg["validation"]["months"] == [6]
assert split_cfg["test"]["months"] == [7]
assert split_cfg["test"]["sealed"] is True

# 3. Create training and validation populations
raw = pd.read_csv(raw_path)

labels = splitting.assign_split(
    raw, config=split_cfg
)

train_df = raw.loc[labels.eq("train")].copy()
val_df = raw.loc[labels.eq("validation")].copy()

# Immediately release raw data and split labels
del raw, labels

target = prep_cfg["target_column"]
month_field = split_cfg["temporal_field"]

# 4. Verify populations and fraud counts
assert len(train_df) == split_cfg["train"]["records"]
assert len(val_df) == split_cfg["validation"]["records"]

assert int(train_df[target].sum()) == (
    split_cfg["train"]["fraud_cases"]
)
assert int(val_df[target].sum()) == (
    split_cfg["validation"]["fraud_cases"]
)

assert set(train_df[month_field].unique()) == set(range(6))
assert set(val_df[month_field].unique()) == {6}

# 5. Prepare the original 34 model features
train_features = preprocessing.prepare_model_features(
    train_df, prep_cfg
)
val_features = preprocessing.prepare_model_features(
    val_df, prep_cfg
)

m06_y_train = train_df[target].to_numpy(
    dtype=np.float32
)
m06_y_validation = val_df[target].to_numpy(
    dtype=np.float32
)

assert train_features.shape[1] == 34
assert val_features.shape[1] == 34
assert train_features.columns.equals(val_features.columns)

assert target not in train_features.columns
assert month_field not in train_features.columns

# Raw split DataFrames are no longer needed
del train_df, val_df

# 6. Load the trusted fitted preprocessor
# DO NOT fit or refit it.
fitted_preprocessor = joblib.load(preprocessor_path)

m06_X_train = np.asarray(
    fitted_preprocessor.transform(train_features),
    dtype=np.float32
)

m06_X_validation = np.asarray(
    fitted_preprocessor.transform(val_features),
    dtype=np.float32
)

m06_feature_names = list(
    fitted_preprocessor.get_feature_names_out()
)

# 7. Verify against Milestone 05 metadata
with open(metadata_path, "r", encoding="utf-8") as file:
    reference_metadata = json.load(file)

assert (
    m06_feature_names
    == reference_metadata["data"]["processed_feature_names"]
)

assert m06_X_train.shape == (794989, 55)
assert m06_X_validation.shape == (108168, 55)

assert m06_y_train.shape == (794989,)
assert m06_y_validation.shape == (108168,)

assert m06_X_train.dtype == np.float32
assert m06_X_validation.dtype == np.float32

assert np.isfinite(m06_X_train).all()
assert np.isfinite(m06_X_validation).all()

assert set(np.unique(m06_y_train)) == {0.0, 1.0}
assert set(np.unique(m06_y_validation)) == {0.0, 1.0}

# 8. Display final verification
print("MILESTONE 06 — PORTABLE DATA PREPARATION")
print("=" * 60)

print("Training features:", m06_X_train.shape)
print("Validation features:", m06_X_validation.shape)
print("Training labels:", m06_y_train.shape)
print("Validation labels:", m06_y_validation.shape)

print("\nTraining fraud cases:", int(m06_y_train.sum()))
print("Validation fraud cases:", int(m06_y_validation.sum()))
print("Feature dtype:", m06_X_train.dtype)
print("Feature count:", len(m06_feature_names))

print("\nFitted preprocessor reused: PASS")
print("Feature order matches Milestone 05: PASS")
print("Training/validation alignment: PASS")
print("Non-finite feature check: PASS")
print("Month-7 test data excluded: PASS")
print("Portable data preparation: PASS")


### Export Portable NumPy Arrays
**What we're doing**

We'll create one compressed .npz archive containing:

**Array	Contents**
- X_train	794,989 × 55 training features
- y_train	794,989 training labels
- X_validation	108,168 × 55 validation features
- y_validation	108,168 validation labels
- feature_names	Exact order of all 55 inputs

In [ ]:

# ============================================================
# Milestone 06 — Cell 9: Export Portable Colab Arrays
# ============================================================

import hashlib
import subprocess

# 1. Define versioned archive path
COLAB_EXPORT_DIR = PROJECT_ROOT / "artifacts" / "milestone_06"

COLAB_ARRAYS_PATH = (
    COLAB_EXPORT_DIR / "colab_dev_arrays_v01.npz"
)

# 2. Verify the file is excluded from Git
relative_path = COLAB_ARRAYS_PATH.relative_to(
    PROJECT_ROOT
).as_posix()

ignore_check = subprocess.run(
    ["git", "check-ignore", "--quiet", relative_path],
    cwd=PROJECT_ROOT,
    check=False,
)

assert ignore_check.returncode == 0, (
    "STOP: Export destination is not confirmed "
    "as Git-ignored."
)

# 3. Validate the arrays before export
assert m06_X_train.shape == (794989, 55)
assert m06_X_validation.shape == (108168, 55)
assert m06_y_train.shape == (794989,)
assert m06_y_validation.shape == (108168,)

assert m06_X_train.dtype == np.float32
assert m06_X_validation.dtype == np.float32

assert len(m06_feature_names) == 55
assert int(m06_y_train.sum()) == 8151
assert int(m06_y_validation.sum()) == 1450

assert split_cfg["test"]["sealed"] is True

# 4. Prevent accidental overwriting
if COLAB_ARRAYS_PATH.exists():
    raise FileExistsError(
        f"Export already exists: {COLAB_ARRAYS_PATH}. "
        "Choose a new version instead."
    )

COLAB_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 5. Export only training and validation data
with COLAB_ARRAYS_PATH.open("xb") as archive:
    np.savez_compressed(
        archive,
        X_train=m06_X_train,
        y_train=m06_y_train,
        X_validation=m06_X_validation,
        y_validation=m06_y_validation,
        feature_names=np.asarray(m06_feature_names),
    )

# 6. Reload and verify the archive
with np.load(
    COLAB_ARRAYS_PATH,
    allow_pickle=False,
) as saved:

    expected_keys = {
        "X_train",
        "y_train",
        "X_validation",
        "y_validation",
        "feature_names",
    }

    assert set(saved.files) == expected_keys

    assert np.array_equal(
        saved["X_train"], m06_X_train
    )
    assert np.array_equal(
        saved["y_train"], m06_y_train
    )
    assert np.array_equal(
        saved["X_validation"], m06_X_validation
    )
    assert np.array_equal(
        saved["y_validation"], m06_y_validation
    )
    assert saved["feature_names"].tolist() == m06_feature_names

# 7. Calculate SHA-256 without loading file into memory
sha256 = hashlib.sha256()

with COLAB_ARRAYS_PATH.open("rb") as file:
    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b"",
    ):
        sha256.update(chunk)

COLAB_ARRAYS_SHA256 = sha256.hexdigest()

# 8. Display export results
print("MILESTONE 06 — COLAB ARRAY EXPORT")
print("=" * 60)

print("Archive:", COLAB_ARRAYS_PATH.name)
print(
    "File size (MiB):",
    round(COLAB_ARRAYS_PATH.stat().st_size / (1024 ** 2), 2)
)
print("SHA-256:", COLAB_ARRAYS_SHA256)

print("\nGit exclusion: PASS")
print("All five arrays verified: PASS")
print("Array contents match source: PASS")
print("Overwrite protection: ENABLED")
print("Month-7 data included: NO")
print("Portable array export: PASS")


### Create the Data Integrity Manifest

In [ ]:

# ============================================================
# Milestone 06 — Cell 10: Portable Data Integrity Manifest
# ============================================================

import platform
import sklearn

# 1. Define the manifest destination
MANIFEST_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "milestone_06"
    / "colab_dev_manifest_v01.json"
)

# 2. Verify the exported archive still exists
assert COLAB_ARRAYS_PATH.is_file()

# Recalculate SHA-256 from the actual file
archive_hash = hashlib.sha256()

with COLAB_ARRAYS_PATH.open("rb") as file:
    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b"",
    ):
        archive_hash.update(chunk)

assert archive_hash.hexdigest() == COLAB_ARRAYS_SHA256

# 3. Record provenance of the fitted preprocessor
preprocessor_hash = hashlib.sha256(
    preprocessor_path.read_bytes()
).hexdigest()

# 4. Build the portable manifest
data_manifest = {
    "schema_version": 1,
    "milestone": "06",
    "archive_name": COLAB_ARRAYS_PATH.name,
    "archive_sha256": COLAB_ARRAYS_SHA256,

    "source": {
        "dataset": "Base.csv",
        "preprocessing": "Milestone 05 fitted preprocessor",
        "preprocessor_sha256": preprocessor_hash,
        "python_version": platform.python_version(),
        "scikit_learn_version": sklearn.__version__,
    },

    "training": {
        "months": split_cfg["train"]["months"],
        "feature_shape": list(m06_X_train.shape),
        "target_shape": list(m06_y_train.shape),
        "fraud_cases": int(m06_y_train.sum()),
    },

    "validation": {
        "months": split_cfg["validation"]["months"],
        "feature_shape": list(m06_X_validation.shape),
        "target_shape": list(m06_y_validation.shape),
        "fraud_cases": int(m06_y_validation.sum()),
    },

    "features": {
        "count": len(m06_feature_names),
        "dtype": str(m06_X_train.dtype),
        "ordered_names": [
            str(name) for name in m06_feature_names
        ],
    },

    "holdout": {
        "month": 7,
        "status": "SEALED",
        "included_in_archive": False,
    },
}

# 5. Validate manifest before saving
assert data_manifest["training"]["feature_shape"] == [794989, 55]
assert data_manifest["validation"]["feature_shape"] == [108168, 55]
assert data_manifest["features"]["count"] == 55
assert data_manifest["training"]["fraud_cases"] == 8151
assert data_manifest["validation"]["fraud_cases"] == 1450
assert data_manifest["holdout"]["included_in_archive"] is False

assert data_manifest["features"]["ordered_names"] == (
    reference_metadata["data"]["processed_feature_names"]
)

# 6. Refuse to overwrite existing metadata
if MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Manifest already exists: {MANIFEST_PATH}. "
        "Use a new versioned filename."
    )

# 7. Save and reload the manifest
with MANIFEST_PATH.open("x", encoding="utf-8") as file:
    json.dump(data_manifest, file, indent=2)
    file.write("\n")

with MANIFEST_PATH.open("r", encoding="utf-8") as file:
    loaded_manifest = json.load(file)

assert loaded_manifest == data_manifest

# 8. Display verification
print("MILESTONE 06 — DATA INTEGRITY MANIFEST")
print("=" * 60)

print("Manifest:", MANIFEST_PATH.name)
print("Archive:", loaded_manifest["archive_name"])
print("Archive SHA-256:", loaded_manifest["archive_sha256"])
print("Feature count:", loaded_manifest["features"]["count"])
print("Training records:", loaded_manifest["training"]["feature_shape"][0])
print("Validation records:", loaded_manifest["validation"]["feature_shape"][0])

print("\nArchive checksum verified: PASS")
print("Feature ordering verified: PASS")
print("Preprocessing provenance recorded: PASS")
print("Manifest save/reload: PASS")
print("Test split: SEALED")
print("Data integrity manifest: PASS")


### Register Both GPU Experiments

In [ ]:

# ============================================================
# Milestone 06 — Register Verified Colab GPU Experiments
# ============================================================

import json
import hashlib
import zipfile
import pandas as pd

# 1. Confirm our original registry is available
assert "experiment_registry" in globals(), (
    "Run the existing experiment registry setup cells first."
)
assert "register_experiment" in globals(), (
    "Run the safe experiment registration cell first."
)

assert len(experiment_registry) == 1
assert experiment_registry[0]["experiment_id"] == (
    "m05_fixed_budget_baseline"
)

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "milestone_06"

# 2. Define the two verified exports
GPU_EXPORTS = [
    {
        "zip": "m06_unweighted_gpu_reference_v01.zip",
        "history": "m06_unweighted_gpu_history_v01.json",
        "manifest": "m06_unweighted_gpu_manifest_v01.json",
        "sha256": "352bdd96315364a7e75d1a0536937343b33d46029f7122dbc8e81927d419b447",
        "imbalance": "none",
        "loss_key": "val_loss",
        "expected_pr_auc": 0.160393,
    },
    {
        "zip": "m06_class_weighted_gpu_v01.zip",
        "history": "m06_class_weighted_history_v01.json",
        "manifest": "m06_class_weighted_manifest_v01.json",
        "sha256": "a426d3e7f07405172d45113a2c3c9b717e5366d3c54ba27259bd894a2ee64b55",
        "imbalance": "positive_class_weight",
        "loss_key": "val_unweighted_loss",
        "expected_pr_auc": 0.163862,
    },
]

# 3. Read and verify each ZIP before registering
def load_gpu_experiment(spec):
    path = ARTIFACT_DIR / spec["zip"]
    assert path.is_file(), f"Missing archive: {path}"

    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    assert digest.hexdigest() == spec["sha256"]

    with zipfile.ZipFile(path) as archive:
        assert archive.testzip() is None

        history = json.loads(
            archive.read(spec["history"])
        )
        manifest = json.loads(
            archive.read(spec["manifest"])
        )

        for filename, expected_hash in manifest["files"].items():
            actual_hash = hashlib.sha256(
                archive.read(filename)
            ).hexdigest()
            assert actual_hash == expected_hash

    # Verify experiment provenance
    assert history["source_commit"] == "cf443a5"
    assert history["test_status"] == "SEALED"
    assert history["train_months"] == [0, 1, 2, 3, 4, 5]
    assert history["validation_months"] == [6]
    assert history["feature_count"] == 55
    assert history["parameter_count"] == 5697
    assert history["epochs"] == 5
    assert history["seed"] == 42
    assert len(history["epoch_history"]) == 5

    final = history["final_validation"]

    assert final["epoch"] == 5
    assert abs(
        final["val_pr_auc"] - spec["expected_pr_auc"]
    ) < 0.000001

    # Both validation losses are UNWEIGHTED and comparable.
    record = {
        "experiment_id": history["experiment_id"],
        "source": "colab_verified_archive",
        "status": "completed_verified",
        "device": history["device"],
        "seed": history["seed"],
        "epochs": history["epochs"],
        "learning_rate": history["learning_rate"],
        "dropout_rate": 0.10,
        "imbalance_strategy": spec["imbalance"],
        "validation_pr_auc": float(final["val_pr_auc"]),
        "validation_roc_auc": float(final["val_roc_auc"]),
        "validation_loss": float(final[spec["loss_key"]]),
        "source_commit": history["source_commit"],
        "artifact_path": str(path.relative_to(PROJECT_ROOT)),
        "artifact_sha256": spec["sha256"],
    }

    return record

# 4. Validate both records in a temporary registry
# The original registry is unchanged if either record fails.
proposed_registry = [
    record.copy() for record in experiment_registry
]

for specification in GPU_EXPORTS:
    record = load_gpu_experiment(specification)
    register_experiment(proposed_registry, record)

# 5. Commit both verified records together
assert len(proposed_registry) == 3

experiment_registry = proposed_registry
registry_df = pd.DataFrame(experiment_registry)

assert registry_df["experiment_id"].is_unique
assert registry_df["validation_pr_auc"].between(0, 1).all()

# 6. Display updated registry
print("MILESTONE 06 — EXPERIMENT REGISTRY")
print("=" * 65)

display(
    registry_df[
        [
            "experiment_id",
            "device",
            "imbalance_strategy",
            "epochs",
            "validation_pr_auc",
            "validation_roc_auc",
            "status",
        ]
    ].round(6)
)

print("\nHistorical M05 baseline preserved: PASS")
print("Unweighted GPU experiment registered: PASS")
print("Class-weighted GPU experiment registered: PASS")
print("Archive integrity verification: PASS")
print("Duplicate experiment protection: PASS")
print("Month-7 test set: SEALED")
print("Step 06.30B: PASS")


### Export Experiment Registry v02

In [ ]:

# ============================================================
# Milestone 06 — Cell: Export Experiment Registry v02
# ============================================================

import json
import hashlib
from pathlib import Path

# 1. Verify the registered experiments
expected_ids = {
    "m05_fixed_budget_baseline",
    "m06_gpu_reference_unweighted_v01",
    "m06_class_weighted_gpu_v01",
}

actual_ids = {
    record["experiment_id"]
    for record in experiment_registry
}

assert len(experiment_registry) == 3
assert actual_ids == expected_ids

# Prevent test metrics from entering the registry
assert all(
    not any(str(key).startswith("test_") for key in record)
    for record in experiment_registry
)

# 2. Prepare a portable registry document
registry_document = {
    "registry_version": 2,
    "project": "bank-account-fraud-dl-mlops",
    "milestone": "06_model_improvement",
    "source_notebook": "notebooks/06_model_improvement.ipynb",
    "primary_metric": "validation_pr_auc",
    "selection_epoch": 5,
    "train_months": [0, 1, 2, 3, 4, 5],
    "validation_months": [6],
    "holdout_status": "SEALED",
    "experiment_count": len(experiment_registry),
    "experiments": [
        record.copy()
        for record in experiment_registry
    ],
}

# 3. Define the versioned output file
export_path = (
    PROJECT_ROOT
    / "artifacts"
    / "milestone_06"
    / "experiment_registry_v02.json"
)

assert export_path.parent.is_dir(), (
    "Milestone 06 artifact directory is missing."
)

# Never overwrite an existing registry snapshot
assert not export_path.exists(), (
    f"Registry v02 already exists: {export_path}"
)

# 4. Export safely
with export_path.open("x", encoding="utf-8") as file:
    json.dump(
        registry_document,
        file,
        indent=2,
        allow_nan=False,
    )

# 5. Independently reload the saved file
with export_path.open("r", encoding="utf-8") as file:
    restored_registry = json.load(file)

assert restored_registry == registry_document

assert restored_registry["experiment_count"] == 3
assert restored_registry["holdout_status"] == "SEALED"

# 6. Generate SHA-256 checksum
sha256 = hashlib.sha256()

with export_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

# 7. Display results
print("MILESTONE 06 — REGISTRY V02 EXPORT")
print("=" * 60)

print("File:", export_path.name)
print("Registered experiments:", len(experiment_registry))

print("\nVALIDATION PR-AUC")

for record in experiment_registry:
    print(
        f"{record['experiment_id']}: "
        f"{record['validation_pr_auc']:.6f}"
    )

print("\nSHA-256:", sha256.hexdigest())

print("\nRegistry v02 created: PASS")
print("JSON reload verification: PASS")
print("Three experiment records: PASS")
print("Original registry v01: PRESERVED")
print("Month-7 test set: SEALED")
print("Step 06.30C: PASS")


### Register the Weight-25 GPU Experiment

In [ ]:

# ============================================================
# Milestone 06 — Step 06.33E: Register Weight-25 Experiment
# ============================================================

import json
import hashlib
import zipfile
import pandas as pd

# 1. Verify the existing registry is available
assert "experiment_registry" in globals(), (
    "Experiment registry not loaded. Stop and restore "
    "the existing registry before continuing."
)

assert "register_experiment" in globals(), (
    "Registration function not loaded."
)

expected_existing_ids = {
    "m05_fixed_budget_baseline",
    "m06_gpu_reference_unweighted_v01",
    "m06_class_weighted_gpu_v01",
}

assert len(experiment_registry) == 3
assert {
    record["experiment_id"]
    for record in experiment_registry
} == expected_existing_ids

# 2. Locate the verified Weight-25 artifact
archive_path = (
    PROJECT_ROOT
    / "artifacts"
    / "milestone_06"
    / "m06_weight25_gpu_v01.zip"
)

assert archive_path.is_file()

expected_zip_sha256 = (
    "644255d89b7d47eb52ae9418e7af0b22"
    "f65817271be703869123f8a3c0547814"
)

digest = hashlib.sha256()

with archive_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        digest.update(chunk)

assert digest.hexdigest() == expected_zip_sha256

# 3. Read archived history and verify internal checksums
with zipfile.ZipFile(archive_path, "r") as archive:

    assert archive.testzip() is None

    assert set(archive.namelist()) == {
        "m06_weight25_gpu_v01.pt",
        "m06_weight25_history_v01.json",
        "m06_weight25_manifest_v01.json",
    }

    history = json.loads(
        archive.read("m06_weight25_history_v01.json")
    )

    manifest = json.loads(
        archive.read("m06_weight25_manifest_v01.json")
    )

    for filename, expected_hash in manifest["files"].items():

        actual_hash = hashlib.sha256(
            archive.read(filename)
        ).hexdigest()

        assert actual_hash == expected_hash

# 4. Verify experiment provenance and configuration
assert history["experiment_id"] == "m06_weight25_gpu_v01"
assert history["source_commit"] == "cf443a5"

assert history["architecture"] == "FraudMLP"
assert history["feature_count"] == 55
assert history["parameter_count"] == 5697

assert history["seed"] == 42
assert history["epochs"] == 5
assert history["batch_size"] == 1024
assert history["learning_rate"] == 0.001
assert history["positive_class_weight"] == 25.0

assert history["train_months"] == [0, 1, 2, 3, 4, 5]
assert history["validation_months"] == [6]
assert history["test_status"] == "SEALED"

assert len(history["epoch_history"]) == 5

# 5. Verify fixed-epoch validation metrics
final = history["final_validation"]

assert final["epoch"] == 5

assert abs(
    final["val_pr_auc"] - 0.161925
) < 0.000001

assert abs(
    final["val_roc_auc"] - 0.885421
) < 0.000001

assert abs(
    final["val_unweighted_loss"] - 0.163284
) < 0.000001

# 6. Construct the registry record
weight25_record = {
    "experiment_id": history["experiment_id"],
    "source": "colab_verified_archive",
    "status": "completed_verified",
    "device": history["device"],
    "seed": history["seed"],
    "epochs": history["epochs"],
    "learning_rate": history["learning_rate"],
    "dropout_rate": 0.10,
    "imbalance_strategy": "positive_class_weight_25",
    "validation_pr_auc": float(final["val_pr_auc"]),
    "validation_roc_auc": float(final["val_roc_auc"]),
    "validation_loss": float(final["val_unweighted_loss"]),
    "source_commit": history["source_commit"],
    "artifact_path": str(
        archive_path.relative_to(PROJECT_ROOT)
    ),
    "artifact_sha256": expected_zip_sha256,
}

# 7. Validate on a copy first
proposed_registry = [
    record.copy()
    for record in experiment_registry
]

register_experiment(
    proposed_registry,
    weight25_record,
)

assert len(proposed_registry) == 4

assert len({
    record["experiment_id"]
    for record in proposed_registry
}) == 4

# 8. Update the in-memory registry only after validation
experiment_registry = proposed_registry
registry_df = pd.DataFrame(experiment_registry)

print("MILESTONE 06 — FOUR-EXPERIMENT REGISTRY")
print("=" * 65)

display(
    registry_df[
        [
            "experiment_id",
            "imbalance_strategy",
            "epochs",
            "validation_pr_auc",
            "validation_roc_auc",
            "status",
        ]
    ].round(6)
)

print("\nWeight-25 archive integrity: PASS")
print("Weight-25 configuration: PASS")
print("Weight-25 experiment registered: PASS")
print("Previous three experiments preserved: PASS")
print("Registry experiment count:", len(experiment_registry))
print("Month-7 test set: SEALED")
print("Step 06.33E: PASS")


### Export Experiment Registry v03

In [ ]:

# ============================================================
# Milestone 06 — Step 06.33F: Export Registry v03
# ============================================================

import json
import hashlib
from pathlib import Path

# 1. Verify the four registered experiments
expected_ids = [
    "m05_fixed_budget_baseline",
    "m06_gpu_reference_unweighted_v01",
    "m06_class_weighted_gpu_v01",
    "m06_weight25_gpu_v01",
]

actual_ids = [
    record["experiment_id"]
    for record in experiment_registry
]

assert actual_ids == expected_ids
assert len(set(actual_ids)) == 4

# 2. Confirm the sealed test set is protected
assert all(
    not any(str(key).startswith("test_") for key in record)
    for record in experiment_registry
)

# 3. Locate existing versioned registries
artifact_dir = (
    PROJECT_ROOT / "artifacts" / "milestone_06"
)

v01_path = artifact_dir / "experiment_registry_v01.json"
v02_path = artifact_dir / "experiment_registry_v02.json"
v03_path = artifact_dir / "experiment_registry_v03.json"

assert v01_path.is_file()
assert v02_path.is_file()
assert not v03_path.exists(), (
    "Registry v03 already exists. Do not overwrite it."
)

# 4. Verify the original three records remain unchanged
with v02_path.open("r", encoding="utf-8") as file:
    previous_registry = json.load(file)

assert previous_registry["registry_version"] == 2
assert previous_registry["experiment_count"] == 3

assert (
    previous_registry["experiments"]
    == experiment_registry[:3]
), "Previous experiment records have changed."

# 5. Build the new registry snapshot
registry_v03 = {
    "registry_version": 3,
    "project": "bank-account-fraud-dl-mlops",
    "milestone": "06_model_improvement",
    "source_notebook": "notebooks/06_model_improvement.ipynb",
    "primary_metric": "validation_pr_auc",
    "selection_epoch": 5,
    "train_months": [0, 1, 2, 3, 4, 5],
    "validation_months": [6],
    "holdout_status": "SEALED",
    "experiment_count": 4,
    "experiments": [
        record.copy()
        for record in experiment_registry
    ],
}

# 6. Save without overwriting an existing file
with v03_path.open("x", encoding="utf-8") as file:
    json.dump(
        registry_v03,
        file,
        indent=2,
        allow_nan=False,
    )

# 7. Reload and verify the exported registry
with v03_path.open("r", encoding="utf-8") as file:
    restored = json.load(file)

assert restored == registry_v03
assert restored["experiment_count"] == 4
assert restored["holdout_status"] == "SEALED"

# 8. Calculate SHA-256 checksum
digest = hashlib.sha256()

with v03_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        digest.update(chunk)

# 9. Display verification results
print("MILESTONE 06 — REGISTRY V03 EXPORT")
print("=" * 60)

print("Filename:", v03_path.name)
print("Registered experiments:", restored["experiment_count"])

print("\nVALIDATION PR-AUC")

for record in restored["experiments"]:
    print(
        f"{record['experiment_id']}: "
        f"{record['validation_pr_auc']:.6f}"
    )

print("\nSHA-256:", digest.hexdigest())

print("\nRegistry v03 created: PASS")
print("JSON reload verification: PASS")
print("Four experiment records: PASS")
print("Previous registry records unchanged: PASS")
print("Registry v01 and v02: PRESERVED")
print("Month-7 test set: SEALED")
print("Step 06.33F: PASS")


### Reproduce the HistGradientBoosting Baseline

In [ ]:

# ============================================================
# Milestone 06 — Reproduce HistGradientBoosting Baseline
# Development data only: train months 0–5, validation month 6
# Month 7 remains sealed.
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

artifact_dir = Path("artifacts/milestone_06")

# Resolve path when working from /notebooks.
if not artifact_dir.exists():
    artifact_dir = Path("../artifacts/milestone_06")

manifest = json.loads(
    (artifact_dir / "colab_dev_manifest_v01.json").read_text(
        encoding="utf-8"
    )
)

# Ensure we use the original preprocessing environment.
print("Current scikit-learn:", sklearn.__version__)
print("Original scikit-learn:", manifest["source"]["scikit_learn_version"])

assert (
    sklearn.__version__
    == manifest["source"]["scikit_learn_version"]
), "scikit-learn version differs; stop before training."

# Load training and validation arrays ONLY.
with np.load(
    artifact_dir / manifest["archive_name"],
    allow_pickle=False,
) as data:
    hgb_X_train = data["X_train"]
    hgb_y_train = data["y_train"].astype(np.int64)

    hgb_X_val = data["X_validation"]
    hgb_y_val = data["y_validation"].astype(np.int64)

    feature_names = data["feature_names"].tolist()

# Verify data integrity before fitting.
assert hgb_X_train.shape == (794989, 55)
assert hgb_X_val.shape == (108168, 55)
assert int(hgb_y_train.sum()) == 8151
assert int(hgb_y_val.sum()) == 1450
assert np.isin(hgb_y_train, [0, 1]).all()
assert np.isin(hgb_y_val, [0, 1]).all()
assert np.isfinite(hgb_X_train).all()
assert np.isfinite(hgb_X_val).all()
assert feature_names == manifest["features"]["ordered_names"]

print("\nDevelopment-data checks: PASS")

# Reproduce the original Milestone 04 configuration.
hgb_model = HistGradientBoostingClassifier(
    learning_rate=0.10,
    max_iter=100,
    max_leaf_nodes=31,
    early_stopping=False,
    random_state=42,
)

print("\nTraining HistGradientBoosting on months 0–5...")

start = time.perf_counter()

hgb_model.fit(hgb_X_train, hgb_y_train)

hgb_fit_seconds = time.perf_counter() - start

# Evaluate ONLY on month 6.
hgb_val_proba = hgb_model.predict_proba(hgb_X_val)[:, 1]

hgb_pr_auc = average_precision_score(
    hgb_y_val,
    hgb_val_proba,
)

historical_pr_auc = 0.166961

print("\nHISTGRADIENTBOOSTING REPRODUCTION RESULTS")
print("=" * 55)
print(f"Historical PR-AUC:  {historical_pr_auc:.6f}")
print(f"Reproduced PR-AUC: {hgb_pr_auc:.6f}")
print(f"Difference:        {hgb_pr_auc - historical_pr_auc:+.6f}")
print(f"Training time:     {hgb_fit_seconds:.2f} seconds")
print(f"Iterations:        {hgb_model.n_iter_}")
print("Validation records:", len(hgb_val_proba))
print("Month-7 test status: SEALED")


### Compare Fraud Detection at Fixed Alert Budgets

In [ ]:

# ============================================================
# Step 06.56 — HistGradientBoosting Alert-Budget Evaluation
# Validation month 6 ONLY — month 7 remains sealed
# ============================================================

import numpy as np

# 1. Verify that the predictions and labels are valid
assert len(hgb_val_proba) == len(hgb_y_val) == 108168
assert int(hgb_y_val.sum()) == 1450
assert np.isfinite(hgb_val_proba).all()
assert np.all((hgb_val_proba >= 0) & (hgb_val_proba <= 1))

# 2. Rank transactions from highest to lowest predicted fraud risk
# Stable sorting ensures deterministic handling of tied scores.
ranked_indices = np.argsort(-hgb_val_proba, kind="stable")

# 3. Previously verified GPU MLP fraud counts
weighted_mlp_reference = {
    "1%": 277,
    "5%": 706,
}

print("HISTGRADIENTBOOSTING — VALIDATION ALERT BUDGETS")
print("=" * 65)

for label, fraction in [("1%", 0.01), ("5%", 0.05)]:

    # Use the same whole-number alert budgets as the GPU models.
    k = int(len(hgb_y_val) * fraction)

    # Select the k highest-risk validation transactions.
    selected_labels = hgb_y_val[ranked_indices[:k]]

    fraud_detected = int(selected_labels.sum())
    false_positives = k - fraud_detected
    fraud_missed = int(hgb_y_val.sum()) - fraud_detected

    precision_at_k = fraud_detected / k
    recall_at_k = fraud_detected / int(hgb_y_val.sum())

    difference = (
        fraud_detected - weighted_mlp_reference[label]
    )

    print(f"\nTop {label} alert budget")
    print(f"Alerts reviewed:      {k:,}")
    print(f"Fraud detected:       {fraud_detected:,}")
    print(f"False positives:      {false_positives:,}")
    print(f"Fraud missed:         {fraud_missed:,}")
    print(f"Precision at K:       {precision_at_k:.6f}")
    print(f"Recall at K:          {recall_at_k:.6f}")
    print(f"Vs. weighted GPU MLP: {difference:+d} fraud cases")

print("\nEvaluation partition: MONTH 6 ONLY")
print("Month-7 test set: SEALED")
print("Step 06.56: PASS")

## **Provisional Decision**
The HistGradientBoosting model has the highest overall PR-AUC, but the fully weighted neural network catches more fraud at our selected investigation budgets.
At a 5% investigation budget, the weighted MLP identifies 26 additional fraudulent transactions compared with HistGradientBoosting, using the same 5,408 alerts.
This demonstrates an important concept: the model with the highest overall PR-AUC does not necessarily perform best at a specific operational alert budget.

The full class-weighted GPU MLP is the leading deep-learning candidate for the capstone, particularly if our objective is detecting the most fraud within a fixed investigation budget.
It is not our final deployment decision yet. We still need to assess calibration, robustness, and operational constraints. These results are from one validation month, and the differences—especially at the 1% budget—may not be robust.

### Preserve the Reproduced HistGradientBoosting Baseline

In [ ]:

# ============================================================
# Step 06.57 — Preserve HistGradientBoosting Baseline
# Training months 0–5; validation month 6 only
# ============================================================

import json
import hashlib
import joblib
import numpy as np
import sklearn

# Use the artifact directory resolved in Step 06.55.
artifact_dir.mkdir(parents=True, exist_ok=True)

model_path = artifact_dir / "m06_hgb_reproduced_v01.joblib"
predictions_path = artifact_dir / "m06_hgb_validation_predictions_v01.npz"
manifest_path = artifact_dir / "m06_hgb_reproduction_v01.json"

# Protect existing artifacts from accidental overwriting.
for path in [model_path, predictions_path, manifest_path]:
    assert not path.exists(), f"Artifact already exists: {path}"

# Verify this is the development-only evaluation.
assert hgb_X_train.shape == (794989, 55)
assert hgb_X_val.shape == (108168, 55)
assert len(hgb_val_proba) == 108168
assert int(hgb_y_val.sum()) == 1450
assert round(float(hgb_pr_auc), 6) == 0.166961

# Save the trained baseline.
joblib.dump(hgb_model, model_path, compress=3)

# Save month-6 validation predictions and matching labels.
np.savez_compressed(
    predictions_path,
    validation_probability=hgb_val_proba,
    validation_target=hgb_y_val.astype(np.int8),
)

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

# Record reproducibility information.
report = {
    "report_id": "m06_hgb_reproduction_v01",
    "model": "HistGradientBoostingClassifier",
    "source_notebook": "04_baseline_models.ipynb",
    "training_months": [0, 1, 2, 3, 4, 5],
    "validation_month": 6,
    "validation_records": 108168,
    "validation_fraud_cases": 1450,
    "validation_pr_auc": float(hgb_pr_auc),
    "training_seconds": float(hgb_fit_seconds),
    "training_iterations": int(hgb_model.n_iter_),
    "scikit_learn_version": sklearn.__version__,
    "parameters": hgb_model.get_params(),
    "model_file": model_path.name,
    "model_sha256": file_sha256(model_path),
    "predictions_file": predictions_path.name,
    "predictions_sha256": file_sha256(predictions_path),
    "holdout_status": "SEALED",
}

manifest_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("HISTGRADIENTBOOSTING ARTIFACT PRESERVATION")
print("=" * 55)
print("Model:", model_path.name)
print("Predictions:", predictions_path.name)
print("Manifest:", manifest_path.name)
print(f"Validation PR-AUC: {hgb_pr_auc:.6f}")
print("Month-7 test status: SEALED")
print("Step 06.57: PASS")


### Verify Saved Artifact Integrity

In [ ]:

# ============================================================
# Step 06.58 — Verify Saved HistGradientBoosting Artifacts
# Read-only verification — validation month 6 only
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
from sklearn.metrics import average_precision_score

folder = Path("artifacts/milestone_06")

if not folder.is_dir():
    folder = Path("../artifacts/milestone_06")

report = json.loads(
    (folder / "m06_hgb_reproduction_v01.json").read_text(
        encoding="utf-8"
    )
)

assert report["training_months"] == [0, 1, 2, 3, 4, 5]
assert report["validation_month"] == 6
assert report["holdout_status"] == "SEALED"

# 1. Verify both saved-file checksums
for filename_key, hash_key in [
    ("model_file", "model_sha256"),
    ("predictions_file", "predictions_sha256"),
]:
    path = folder / report[filename_key]

    assert path.is_file(), f"Missing artifact: {path}"

    actual_hash = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    assert actual_hash == report[hash_key], (
        f"Checksum mismatch: {path.name}"
    )

    print(f"{path.name}: SHA256 PASS")

# 2. Reload ONLY month-6 validation predictions
with np.load(
    folder / report["predictions_file"],
    allow_pickle=False,
) as saved:
    assert set(saved.files) == {
        "validation_probability",
        "validation_target",
    }

    probabilities = saved["validation_probability"]
    labels = saved["validation_target"]

assert probabilities.shape == (108168,)
assert labels.shape == (108168,)
assert int(labels.sum()) == 1450
assert np.isin(labels, [0, 1]).all()
assert np.isfinite(probabilities).all()
assert np.all((probabilities >= 0) & (probabilities <= 1))

# 3. Recalculate validation PR-AUC
verified_pr_auc = average_precision_score(
    labels,
    probabilities,
)

assert np.isclose(
    verified_pr_auc,
    report["validation_pr_auc"],
    atol=1e-12,
    rtol=0,
)

# 4. Independently verify both alert budgets
ranked = np.argsort(-probabilities, kind="stable")

for budget, expected_fraud in [
    (0.01, 275),
    (0.05, 680),
]:
    k = int(len(labels) * budget)

    detected = int(labels[ranked[:k]].sum())

    assert detected == expected_fraud, (
        f"Unexpected result at {budget:.0%} budget"
    )

    print(
        f"Top {budget:.0%}: "
        f"{detected} fraud cases / {k:,} alerts — PASS"
    )

print(f"Validation PR-AUC: {verified_pr_auc:.6f}")
print("Saved model checksum: VERIFIED")
print("Saved predictions: VERIFIED")
print("Month-7 test status: SEALED")
print("Step 06.58: PASS")


### Reconstruct the Weighted MLP Predictions

In [ ]:

# ============================================================
# Step 06.61 — Reconstruct Full-Weighted GPU MLP Predictions
# CPU inference only — validation month 6
# Month 7 remains sealed
# ============================================================

from pathlib import Path
from io import BytesIO
import sys
import json
import hashlib
import zipfile

import numpy as np
import torch
from sklearn.metrics import average_precision_score, roc_auc_score

# 1. Resolve project paths
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.mlp import FraudMLP

archive_path = (
    project_root
    / "artifacts"
    / "milestone_06"
    / "m06_class_weighted_gpu_v01.zip"
)

assert archive_path.is_file()

# 2. Verify original ZIP checksum
expected_archive_sha = (
    "a426d3e7f07405172d45113a2c3c9b717e5366d3c54ba27259bd894a2ee64b55"
)

actual_archive_sha = hashlib.sha256(
    archive_path.read_bytes()
).hexdigest()

assert actual_archive_sha == expected_archive_sha

# 3. Verify and read the saved checkpoint
with zipfile.ZipFile(archive_path) as archive:

    assert archive.testzip() is None

    manifest = json.loads(
        archive.read("m06_class_weighted_manifest_v01.json")
    )

    for filename, expected_sha in manifest["files"].items():
        actual_sha = hashlib.sha256(
            archive.read(filename)
        ).hexdigest()

        assert actual_sha == expected_sha, (
            f"Checksum mismatch: {filename}"
        )

    checkpoint_bytes = archive.read(
        "m06_class_weighted_gpu_v01.pt"
    )

# 4. Reconstruct original neural-network architecture
weighted_mlp = FraudMLP(
    input_dim=55,
    dropout_rate=0.10,
)

assert sum(
    p.numel() for p in weighted_mlp.parameters()
) == 5697

# Load weights only, not an arbitrary serialized Python object.
state_dict = torch.load(
    BytesIO(checkpoint_bytes),
    map_location="cpu",
    weights_only=True,
)

weighted_mlp.load_state_dict(
    state_dict,
    strict=True,
)

weighted_mlp.eval()

print("Checkpoint integrity: PASS")
print("Model architecture: PASS")

# 5. Use existing month-6 validation arrays
assert hgb_X_val.shape == (108168, 55)
assert hgb_y_val.shape == (108168,)
assert int(hgb_y_val.sum()) == 1450

# 6. Generate fraud probabilities in small CPU batches
batch_size = 4096
probability_batches = []

with torch.inference_mode():

    for start in range(0, len(hgb_X_val), batch_size):

        end = start + batch_size

        batch = torch.from_numpy(
            np.asarray(
                hgb_X_val[start:end],
                dtype=np.float32,
            )
        )

        logits = weighted_mlp(batch)

        probabilities = torch.sigmoid(
            logits
        ).cpu().numpy().reshape(-1)

        probability_batches.append(probabilities)

weighted_mlp_val_proba = np.concatenate(
    probability_batches
)

assert weighted_mlp_val_proba.shape == (108168,)
assert np.isfinite(weighted_mlp_val_proba).all()

# 7. Recalculate ranking metrics
weighted_mlp_pr_auc = average_precision_score(
    hgb_y_val,
    weighted_mlp_val_proba,
)

weighted_mlp_roc_auc = roc_auc_score(
    hgb_y_val,
    weighted_mlp_val_proba,
)

# 8. Recalculate fixed-budget fraud detection
ranked = np.argsort(
    -weighted_mlp_val_proba,
    kind="stable",
)

topk_results = {}

for budget in (0.01, 0.05):

    k = int(len(hgb_y_val) * budget)

    topk_results[budget] = int(
        hgb_y_val[ranked[:k]].sum()
    )

# 9. Report reconstructed results
print("\nWEIGHTED GPU MLP — CHECKPOINT REPRODUCTION")
print("=" * 60)

print(f"Recorded PR-AUC:      {0.16386246675587196:.9f}")
print(f"Reconstructed PR-AUC: {weighted_mlp_pr_auc:.9f}")

print(f"Recorded ROC-AUC:      {0.8869231316049841:.9f}")
print(f"Reconstructed ROC-AUC: {weighted_mlp_roc_auc:.9f}")

print(f"Top 1% fraud caught: {topk_results[0.01]} / expected 277")
print(f"Top 5% fraud caught: {topk_results[0.05]} / expected 706")

# Small floating-point differences between CPU and GPU
# inference are possible.
metrics_match = (
    abs(weighted_mlp_pr_auc - 0.16386246675587196) < 1e-5
    and abs(weighted_mlp_roc_auc - 0.8869231316049841) < 1e-5
)

topk_match = (
    topk_results[0.01] == 277
    and topk_results[0.05] == 706
)

print("Ranking metrics:", "PASS" if metrics_match else "REVIEW")
print("Alert-budget results:", "PASS" if topk_match else "REVIEW")
print("Month-7 test set: SEALED")

if metrics_match and topk_match:
    print("Step 06.61: PASS")
else:
    print("Step 06.61: REVIEW REQUIRED — do not save predictions yet")


### Save the Weighted MLP Validation Predictions

In [ ]:

# ============================================================
# Step 06.62 — Save Weighted MLP Validation Predictions
# Validation month 6 ONLY — Month 7 remains sealed
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

# 1. Locate our existing local artifacts
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

folder = root / "artifacts" / "milestone_06"
assert folder.is_dir()

hgb_manifest_path = folder / "m06_hgb_reproduction_v01.json"
hgb_report = json.loads(
    hgb_manifest_path.read_text(encoding="utf-8")
)

hgb_predictions_path = folder / hgb_report["predictions_file"]

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

# 2. Verify the saved HGB predictions before comparing labels
assert (
    sha256_file(hgb_predictions_path)
    == hgb_report["predictions_sha256"]
), "HistGradientBoosting predictions checksum mismatch."

with np.load(hgb_predictions_path, allow_pickle=False) as saved_hgb:
    hgb_saved_labels = saved_hgb["validation_target"]

labels = np.asarray(hgb_y_val, dtype=np.int8)
probabilities = np.asarray(weighted_mlp_val_proba)

assert np.array_equal(
    labels,
    hgb_saved_labels,
), "Validation label alignment failed."

assert labels.shape == (108168,)
assert probabilities.shape == (108168,)
assert int(labels.sum()) == 1450
assert np.isfinite(probabilities).all()
assert np.all(
    (probabilities >= 0) & (probabilities <= 1)
)

print("HGB / MLP validation label alignment: PASS")

# 3. Independently verify ranking metrics
pr_auc = average_precision_score(labels, probabilities)
roc_auc = roc_auc_score(labels, probabilities)

assert abs(pr_auc - 0.16386246675587196) < 1e-5
assert abs(roc_auc - 0.8869231316049841) < 1e-5

ranking = np.argsort(-probabilities, kind="stable")

assert int(labels[ranking[:1081]].sum()) == 277
assert int(labels[ranking[:5408]].sum()) == 706

print("MLP ranking metrics: PASS")
print("MLP alert-budget results: PASS")

# 4. Define protected output files
predictions_path = (
    folder / "m06_weighted_mlp_validation_predictions_v01.npz"
)

manifest_path = (
    folder / "m06_weighted_mlp_predictions_manifest_v01.json"
)

assert not predictions_path.exists(), (
    f"Existing file must not be overwritten: {predictions_path}"
)
assert not manifest_path.exists(), (
    f"Existing file must not be overwritten: {manifest_path}"
)

# 5. Save validation probabilities and matching labels
np.savez_compressed(
    predictions_path,
    validation_probability=probabilities,
    validation_target=labels,
)

# 6. Create reproducibility manifest
checkpoint_path = folder / "m06_class_weighted_gpu_v01.zip"

checkpoint_sha = (
    "a426d3e7f07405172d45113a2c3c9b717e5366d3c54ba27259bd894a2ee64b55"
)

assert sha256_file(checkpoint_path) == checkpoint_sha

report = {
    "report_id": "m06_weighted_mlp_predictions_v01",
    "model": "FraudMLP",
    "source_commit": "cf443a546ff655dfcdc01db72d57ae16d022b994",
    "checkpoint_file": checkpoint_path.name,
    "checkpoint_sha256": checkpoint_sha,
    "inference_device": "cpu",
    "training_months": [0, 1, 2, 3, 4, 5],
    "validation_month": 6,
    "validation_records": 108168,
    "validation_fraud_cases": 1450,
    "validation_pr_auc": float(pr_auc),
    "validation_roc_auc": float(roc_auc),
    "top_1_percent_fraud_detected": 277,
    "top_5_percent_fraud_detected": 706,
    "predictions_file": predictions_path.name,
    "predictions_sha256": sha256_file(predictions_path),
    "hgb_reference_predictions_sha256": (
        hgb_report["predictions_sha256"]
    ),
    "holdout_status": "SEALED",
}

manifest_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

# 7. Confirm saved artifacts
print("\nWEIGHTED MLP PREDICTIONS SAVED")
print("=" * 55)
print("Predictions:", predictions_path.name)
print("Manifest:", manifest_path.name)
print(f"Validation PR-AUC: {pr_auc:.9f}")
print(f"Validation ROC-AUC: {roc_auc:.9f}")
print("Top 1% fraud detected: 277")
print("Top 5% fraud detected: 706")
print("Month-7 test set: SEALED")
print("Step 06.62: PASS")


### Independently Verify Both Prediction Files

In [ ]:

# ============================================================
# Step 06.63 — Verify Paired Model Predictions
# Validation month 6 ONLY — Month 7 SEALED
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
from sklearn.metrics import average_precision_score

root = Path.cwd().resolve()

if root.name == "notebooks":
    root = root.parent

folder = root / "artifacts" / "milestone_06"

# 1. Identify both saved manifests
model_files = {
    "HGB": "m06_hgb_reproduction_v01.json",
    "Weighted MLP": "m06_weighted_mlp_predictions_manifest_v01.json",
}

paired_predictions = {}
paired_manifests = {}

# 2. Independently reload and verify each prediction file
for model_name, manifest_filename in model_files.items():

    manifest = json.loads(
        (folder / manifest_filename).read_text(encoding="utf-8")
    )

    assert manifest["validation_month"] == 6
    assert manifest["holdout_status"] == "SEALED"

    prediction_path = folder / manifest["predictions_file"]

    actual_sha = hashlib.sha256(
        prediction_path.read_bytes()
    ).hexdigest()

    assert actual_sha == manifest["predictions_sha256"], (
        f"{model_name}: prediction checksum mismatch"
    )

    with np.load(prediction_path, allow_pickle=False) as saved:
        assert set(saved.files) == {
            "validation_probability",
            "validation_target",
        }

        probabilities = saved["validation_probability"].copy()
        labels = saved["validation_target"].copy()

    assert probabilities.shape == (108168,)
    assert labels.shape == (108168,)
    assert int(labels.sum()) == 1450
    assert np.isin(labels, [0, 1]).all()
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

    pr_auc = average_precision_score(labels, probabilities)

    assert np.isclose(
        pr_auc,
        manifest["validation_pr_auc"],
        atol=1e-12,
        rtol=0,
    )

    paired_predictions[model_name] = {
        "probabilities": probabilities,
        "labels": labels,
    }

    paired_manifests[model_name] = manifest

    print(f"{model_name}: checksum and PR-AUC PASS")

# 3. Confirm identical validation target ordering
paired_validation_labels = paired_predictions["HGB"]["labels"]

assert np.array_equal(
    paired_validation_labels,
    paired_predictions["Weighted MLP"]["labels"],
), "Model validation labels do not align."

print("Validation label alignment: PASS")

# 4. Recalculate top-K results from saved predictions
expected = {
    "HGB": {0.01: 275, 0.05: 680},
    "Weighted MLP": {0.01: 277, 0.05: 706},
}

for model_name, reference in expected.items():

    probabilities = paired_predictions[model_name]["probabilities"]

    ranked = np.argsort(-probabilities, kind="stable")

    for budget, expected_detected in reference.items():

        k = int(len(paired_validation_labels) * budget)

        detected = int(
            paired_validation_labels[ranked[:k]].sum()
        )

        assert detected == expected_detected, (
            f"{model_name}: top-{budget:.0%} mismatch"
        )

        print(
            f"{model_name} top {budget:.0%}: "
            f"{detected} fraud cases — PASS"
        )

print("\nPaired validation predictions: VERIFIED")
print("Month-7 test set: SEALED")
print("Step 06.63: PASS")


### Paired Bootstrap Robustness Analysis

In [ ]:

# ============================================================
# Step 06.64 — Paired Bootstrap Robustness Analysis
# Month 6 validation ONLY — Month 7 remains SEALED
# No retraining and no file modifications
# ============================================================

import time
import numpy as np

# 1. Reuse independently verified paired predictions
labels = np.asarray(
    paired_predictions["HGB"]["labels"],
    dtype=np.int8,
)

hgb_scores = np.asarray(
    paired_predictions["HGB"]["probabilities"],
    dtype=np.float64,
)

mlp_scores = np.asarray(
    paired_predictions["Weighted MLP"]["probabilities"],
    dtype=np.float64,
)

n = len(labels)

assert n == 108168
assert int(labels.sum()) == 1450
assert hgb_scores.shape == mlp_scores.shape == labels.shape

# 2. Define identical investigation budgets
budgets = {
    "1%": int(n * 0.01),
    "5%": int(n * 0.05),
}

def fraud_detected_at_k(y, scores, k):
    """Count fraud cases among the k highest-ranked transactions."""
    ranked = np.argsort(-scores, kind="stable")
    return int(y[ranked[:k]].sum())

# 3. Verify observed results before resampling
observed = {}

for name, k in budgets.items():
    hgb_caught = fraud_detected_at_k(labels, hgb_scores, k)
    mlp_caught = fraud_detected_at_k(labels, mlp_scores, k)

    observed[name] = mlp_caught - hgb_caught

assert observed == {"1%": 2, "5%": 26}

print("Original paired validation results: PASS")

# 4. Paired bootstrap: sample the same rows for both models
N_BOOTSTRAPS = 500
rng = np.random.default_rng(42)

bootstrap_differences = {
    name: np.empty(N_BOOTSTRAPS, dtype=np.int32)
    for name in budgets
}

start_time = time.perf_counter()

for repeat in range(N_BOOTSTRAPS):

    # Sample validation transaction positions with replacement.
    indices = rng.integers(0, n, size=n)

    sampled_labels = labels[indices]
    sampled_hgb = hgb_scores[indices]
    sampled_mlp = mlp_scores[indices]

    # Rank each model independently on the SAME sampled rows.
    hgb_ranked = np.argsort(-sampled_hgb, kind="stable")
    mlp_ranked = np.argsort(-sampled_mlp, kind="stable")

    for name, k in budgets.items():

        hgb_caught = int(
            sampled_labels[hgb_ranked[:k]].sum()
        )

        mlp_caught = int(
            sampled_labels[mlp_ranked[:k]].sum()
        )

        bootstrap_differences[name][repeat] = (
            mlp_caught - hgb_caught
        )

elapsed = time.perf_counter() - start_time

# 5. Summarize the uncertainty in the model differences
print("\nPAIRED BOOTSTRAP ROBUSTNESS RESULTS")
print("=" * 60)
print("Validation month: 6")
print(f"Transactions per resample: {n:,}")
print(f"Bootstrap repetitions: {N_BOOTSTRAPS}")
print("Random seed: 42")

for name, differences in bootstrap_differences.items():

    lower, upper = np.quantile(
        differences,
        [0.025, 0.975],
    )

    favor_mlp = np.mean(differences > 0)
    favor_hgb = np.mean(differences < 0)
    tied = np.mean(differences == 0)

    print(f"\nTop {name} investigation budget")
    print(f"Observed MLP advantage: {observed[name]:+d} fraud")
    print(
        "95% bootstrap percentile interval: "
        f"[{lower:+.1f}, {upper:+.1f}] fraud"
    )
    print(f"Resamples favoring MLP: {favor_mlp:.1%}")
    print(f"Resamples favoring HGB: {favor_hgb:.1%}")
    print(f"Resamples tied:         {tied:.1%}")

print(f"\nBootstrap runtime: {elapsed:.2f} seconds")
print("Month-7 test set: SEALED")
print("Step 06.64: PASS")


## Summary
These results give us our strongest evidence so far about which model performs better under a fixed fraud-investigation budget.
The full class-weighted GPU MLP has a promising advantage at the 5% alert budget, but the evidence at 1% is inconclusive.

### **How we should interpret this**

- **At the 1% alert budget:** The interval crosses zero substantially. We cannot confidently distinguish the models' operational performance at this budget.

- **At the 5% alert budget:** The bootstrap interval is entirely positive, and 97.4% of resamples favor the weighted MLP. That's encouraging evidence that its advantage is not merely an artifact of one particular set of transactions.

However, the lower bound is only +0.5 fraud cases. With 500 bootstrap repetitions, that boundary could shift if we repeat the analysis with more resamples. We should not declare a statistically definitive winner yet.
There's also an important limitation: we've compared and developed several models on month 6. That makes these results exploratory rather than an unbiased estimate of future production performance.

## Our current model-selection position

Provisional DL champion: Full class-weighted PyTorch MLP.
Overall PR-AUC leader: HistGradientBoosting.
Operational leader at the 5% validation alert budget: Full class-weighted MLP.
This is a valuable capstone finding: a model can have slightly lower overall PR-AUC yet detect more fraud within a specific investigation capacity.

### Preserve the Bootstrap Results

In [ ]:

# ============================================================
# Step 06.65 — Preserve Paired Bootstrap Results
# Validation month 6 ONLY — month 7 remains SEALED
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np

root = Path.cwd().resolve()

if root.name == "notebooks":
    root = root.parent

folder = root / "artifacts" / "milestone_06"

assert folder.is_dir()

# 1. Verify the completed bootstrap results
assert N_BOOTSTRAPS == 500
assert observed == {"1%": 2, "5%": 26}

assert set(bootstrap_differences) == {"1%", "5%"}

for values in bootstrap_differences.values():
    assert len(values) == 500
    assert np.isfinite(values).all()

# 2. Protect existing artifacts
distribution_path = (
    folder / "m06_paired_bootstrap_v01.npz"
)

report_path = (
    folder / "m06_paired_bootstrap_report_v01.json"
)

assert not distribution_path.exists(), (
    f"File already exists: {distribution_path}"
)

assert not report_path.exists(), (
    f"File already exists: {report_path}"
)

# 3. Save the complete bootstrap distributions
np.savez_compressed(
    distribution_path,
    difference_top_1pct=bootstrap_differences["1%"],
    difference_top_5pct=bootstrap_differences["5%"],
)

distribution_sha = hashlib.sha256(
    distribution_path.read_bytes()
).hexdigest()

# 4. Summarize the statistical results
summary = {}

for budget, values in bootstrap_differences.items():

    lower, upper = np.quantile(
        values,
        [0.025, 0.975],
    )

    summary[budget] = {
        "observed_mlp_minus_hgb": int(observed[budget]),
        "ci_95_lower": float(lower),
        "ci_95_upper": float(upper),
        "fraction_favoring_mlp": float(np.mean(values > 0)),
        "fraction_favoring_hgb": float(np.mean(values < 0)),
        "fraction_tied": float(np.mean(values == 0)),
    }

# 5. Record methodology and source artifacts
report = {
    "report_id": "m06_paired_bootstrap_v01",
    "validation_month": 6,
    "validation_records": 108168,
    "validation_fraud_cases": 1450,
    "models": [
        "HistGradientBoosting",
        "Full-weighted GPU MLP",
    ],
    "method": "paired_transaction_bootstrap",
    "bootstrap_repetitions": N_BOOTSTRAPS,
    "random_seed": 42,
    "confidence_interval": "95_percentile",
    "results": summary,
    "distribution_file": distribution_path.name,
    "distribution_sha256": distribution_sha,
    "limitations": [
        "Single validation month",
        "Transaction-level independence assumption",
        "Exploratory model selection on validation data",
        "Confidence interval estimated using 500 resamples",
    ],
    "holdout_status": "SEALED",
}

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

# 6. Confirm preservation
print("PAIRED BOOTSTRAP ARTIFACT PRESERVATION")
print("=" * 55)
print("Distributions:", distribution_path.name)
print("Report:", report_path.name)

for budget, result in summary.items():
    print(
        f"Top {budget}: "
        f"observed {result['observed_mlp_minus_hgb']:+d}, "
        f"95% CI "
        f"[{result['ci_95_lower']:+.1f}, "
        f"{result['ci_95_upper']:+.1f}]"
    )

print("Month-7 test set: SEALED")
print("Step 06.65: PASS")


### Verify Saved Bootstrap Artifacts

In [ ]:

# ============================================================
# Step 06.66 — Independent Bootstrap Artifact Verification
# Validation month 6 ONLY — Month 7 SEALED
# Read-only: no training or file modifications
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np

# 1. Resolve artifact directory
root = Path.cwd().resolve()

if root.name == "notebooks":
    root = root.parent

folder = root / "artifacts" / "milestone_06"

report_path = folder / "m06_paired_bootstrap_report_v01.json"

assert report_path.is_file(), "Bootstrap report missing."

report = json.loads(
    report_path.read_text(encoding="utf-8")
)

# 2. Verify experiment provenance
assert report["validation_month"] == 6
assert report["validation_records"] == 108168
assert report["validation_fraud_cases"] == 1450
assert report["holdout_status"] == "SEALED"
assert report["bootstrap_repetitions"] == 500
assert report["random_seed"] == 42
assert report["method"] == "paired_transaction_bootstrap"

print("Experiment provenance: PASS")

# 3. Independently verify saved distribution checksum
distribution_path = folder / report["distribution_file"]

assert distribution_path.is_file()

actual_sha256 = hashlib.sha256(
    distribution_path.read_bytes()
).hexdigest()

assert actual_sha256 == report["distribution_sha256"], (
    "Bootstrap distribution checksum mismatch."
)

print("Bootstrap distribution SHA256: PASS")

# 4. Reload both distributions
with np.load(distribution_path, allow_pickle=False) as saved:

    assert set(saved.files) == {
        "difference_top_1pct",
        "difference_top_5pct",
    }

    saved_differences = {
        "1%": saved["difference_top_1pct"].copy(),
        "5%": saved["difference_top_5pct"].copy(),
    }

# 5. Recalculate statistics independently
expected_observed = {
    "1%": 2,
    "5%": 26,
}

print("\nINDEPENDENT BOOTSTRAP VERIFICATION")
print("=" * 60)

for budget, differences in saved_differences.items():

    assert differences.shape == (500,)
    assert np.issubdtype(differences.dtype, np.integer)

    saved_summary = report["results"][budget]

    lower, upper = np.quantile(
        differences,
        [0.025, 0.975],
    )

    favor_mlp = np.mean(differences > 0)
    favor_hgb = np.mean(differences < 0)
    tied = np.mean(differences == 0)

    assert saved_summary["observed_mlp_minus_hgb"] == (
        expected_observed[budget]
    )

    comparisons = {
        "ci_95_lower": lower,
        "ci_95_upper": upper,
        "fraction_favoring_mlp": favor_mlp,
        "fraction_favoring_hgb": favor_hgb,
        "fraction_tied": tied,
    }

    for metric, recalculated in comparisons.items():
        assert np.isclose(
            recalculated,
            saved_summary[metric],
            atol=1e-12,
            rtol=0,
        ), f"{budget}: mismatch in {metric}"

    print(f"\nTop {budget} alert budget: PASS")
    print(f"Observed MLP advantage: {expected_observed[budget]:+d}")
    print(f"95% interval: [{lower:+.1f}, {upper:+.1f}]")
    print(f"Resamples favoring MLP: {favor_mlp:.1%}")
    print(f"Resamples favoring HGB: {favor_hgb:.1%}")
    print(f"Tied resamples: {tied:.1%}")

print("\nSaved bootstrap results: VERIFIED")
print("Month-7 test set: SEALED")
print("Step 06.66: PASS")


### Expanded Paired Bootstrap (2,000 Resamples)

In [ ]:

# ============================================================
# Step 06.67 — Expanded Paired Bootstrap: 2,000 Resamples
# Training: NONE
# Evaluation: Month 6 validation ONLY
# Month 7: SEALED
# File modifications: NONE
# ============================================================

import time
from pathlib import Path
import numpy as np

# 1. Reuse independently verified validation predictions
labels = np.asarray(
    paired_predictions["HGB"]["labels"],
    dtype=np.int8,
)

hgb_scores = np.asarray(
    paired_predictions["HGB"]["probabilities"],
    dtype=np.float64,
)

mlp_scores = np.asarray(
    paired_predictions["Weighted MLP"]["probabilities"],
    dtype=np.float64,
)

n = len(labels)

assert n == 108168
assert int(labels.sum()) == 1450
assert hgb_scores.shape == (n,)
assert mlp_scores.shape == (n,)

assert np.array_equal(
    labels,
    paired_predictions["Weighted MLP"]["labels"],
)

assert np.isfinite(hgb_scores).all()
assert np.isfinite(mlp_scores).all()

print("Paired validation data: PASS")

# 2. Fixed alert budgets
budgets_v02 = {
    "1%": 1081,
    "5%": 5408,
}

# 3. Verify the observed differences
hgb_original_rank = np.argsort(
    -hgb_scores, kind="stable"
)

mlp_original_rank = np.argsort(
    -mlp_scores, kind="stable"
)

observed_v02 = {}

for name, k in budgets_v02.items():

    hgb_caught = int(
        labels[hgb_original_rank[:k]].sum()
    )

    mlp_caught = int(
        labels[mlp_original_rank[:k]].sum()
    )

    observed_v02[name] = mlp_caught - hgb_caught

assert observed_v02 == {
    "1%": 2,
    "5%": 26,
}

print("Original top-K results: PASS")

# 4. Initialize expanded bootstrap
N_BOOTSTRAPS_V02 = 2000
SEED_V02 = 42

rng_v02 = np.random.default_rng(SEED_V02)

bootstrap_differences_v02 = {
    name: np.empty(N_BOOTSTRAPS_V02, dtype=np.int32)
    for name in budgets_v02
}

print("\nEXPANDED PAIRED BOOTSTRAP")
print("=" * 60)
print("Repetitions:", N_BOOTSTRAPS_V02)
print("Random seed:", SEED_V02)
print("Validation transactions:", n)
print("Training required: NO")

start_time = time.perf_counter()

# 5. Resample identical transaction positions for both models
for repeat in range(N_BOOTSTRAPS_V02):

    indices = rng_v02.integers(
        0, n, size=n
    )

    sampled_labels = labels[indices]

    sampled_hgb = hgb_scores[indices]
    sampled_mlp = mlp_scores[indices]

    # Independently rank scores on the same sampled rows
    hgb_ranked = np.argsort(
        -sampled_hgb, kind="stable"
    )

    mlp_ranked = np.argsort(
        -sampled_mlp, kind="stable"
    )

    for name, k in budgets_v02.items():

        hgb_detected = int(
            sampled_labels[hgb_ranked[:k]].sum()
        )

        mlp_detected = int(
            sampled_labels[mlp_ranked[:k]].sum()
        )

        bootstrap_differences_v02[name][repeat] = (
            mlp_detected - hgb_detected
        )

    # Display progress every 500 repetitions
    if (repeat + 1) % 500 == 0:
        print(
            f"Completed {repeat + 1:,} / "
            f"{N_BOOTSTRAPS_V02:,} resamples",
            flush=True,
        )

elapsed_v02 = time.perf_counter() - start_time

# 6. Verify deterministic reproduction of the first 500
root = Path.cwd().resolve()

if root.name == "notebooks":
    root = root.parent

old_path = (
    root / "artifacts" / "milestone_06"
    / "m06_paired_bootstrap_v01.npz"
)

with np.load(old_path, allow_pickle=False) as old:

    assert np.array_equal(
        bootstrap_differences_v02["1%"][:500],
        old["difference_top_1pct"],
    ), "Top 1% original bootstrap replication failed."

    assert np.array_equal(
        bootstrap_differences_v02["5%"][:500],
        old["difference_top_5pct"],
    ), "Top 5% original bootstrap replication failed."

print("\nFirst 500 resamples match original run: PASS")

# 7. Report expanded results
print("\nEXPANDED BOOTSTRAP RESULTS")
print("=" * 60)

for name, values in bootstrap_differences_v02.items():

    lower, upper = np.quantile(
        values, [0.025, 0.975]
    )

    favor_mlp = np.mean(values > 0)
    favor_hgb = np.mean(values < 0)
    ties = np.mean(values == 0)

    print(f"\nTop {name} alert budget")
    print(
        f"Observed MLP advantage: "
        f"{observed_v02[name]:+d} fraud"
    )
    print(
        f"95% interval: "
        f"[{lower:+.1f}, {upper:+.1f}]"
    )
    print(f"Resamples favoring MLP: {favor_mlp:.1%}")
    print(f"Resamples favoring HGB: {favor_hgb:.1%}")
    print(f"Resamples tied: {ties:.1%}")

    if lower > 0:
        print("Interval: entirely positive")
    elif upper < 0:
        print("Interval: entirely negative")
    else:
        print("Interval: includes zero")

print(f"\nRuntime: {elapsed_v02:.2f} seconds")
print("Month-7 test set: SEALED")
print("Step 06.67: PASS")


## Interpretation
At the 1% budget, the interval includes zero. Neither model has demonstrated a reliable advantage.
At the 5% budget, the weighted MLP catches 706 fraud cases versus 680 for HistGradientBoosting. The expanded bootstrap interval remains entirely positive.
This strengthens our provisional recommendation to use the full-weighted MLP for a 5% alert-budget objective. However, the lower bound is close to zero, and this analysis uses one repeatedly examined validation month. It is not proof of superiority on future data.
We should now preserve these results before proceeding with final model selection.

### Save the Expanded Bootstrap Results

In [ ]:

# ============================================================
# Step 06.68 — Preserve Expanded Bootstrap Results
# Validation month 6 ONLY — Month 7 remains SEALED
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np

# 1. Resolve local artifact directory
root = Path.cwd().resolve()

if root.name == "notebooks":
    root = root.parent

folder = root / "artifacts" / "milestone_06"

assert folder.is_dir()

# 2. Verify completed analysis before saving
assert N_BOOTSTRAPS_V02 == 2000
assert SEED_V02 == 42
assert observed_v02 == {"1%": 2, "5%": 26}
assert set(bootstrap_differences_v02) == {"1%", "5%"}

for budget, values in bootstrap_differences_v02.items():
    assert values.shape == (2000,)
    assert np.issubdtype(values.dtype, np.integer)
    assert np.isfinite(values).all()

# 3. Verify first 500 match the preserved original
old_path = folder / "m06_paired_bootstrap_v01.npz"

with np.load(old_path, allow_pickle=False) as old:
    assert np.array_equal(
        bootstrap_differences_v02["1%"][:500],
        old["difference_top_1pct"],
    )

    assert np.array_equal(
        bootstrap_differences_v02["5%"][:500],
        old["difference_top_5pct"],
    )

print("Original 500-resample reproduction: PASS")

# 4. Define protected output paths
distribution_path = (
    folder / "m06_paired_bootstrap_v02.npz"
)

report_path = (
    folder / "m06_paired_bootstrap_report_v02.json"
)

assert not distribution_path.exists(), (
    f"Existing artifact: {distribution_path}"
)

assert not report_path.exists(), (
    f"Existing artifact: {report_path}"
)

# 5. Calculate and preserve summary statistics
summary = {}

for budget, values in bootstrap_differences_v02.items():
    lower, upper = np.quantile(
        values, [0.025, 0.975]
    )

    summary[budget] = {
        "observed_mlp_minus_hgb": int(observed_v02[budget]),
        "ci_95_lower": float(lower),
        "ci_95_upper": float(upper),
        "fraction_favoring_mlp": float(np.mean(values > 0)),
        "fraction_favoring_hgb": float(np.mean(values < 0)),
        "fraction_tied": float(np.mean(values == 0)),
    }

# 6. Save complete bootstrap distributions
np.savez_compressed(
    distribution_path,
    difference_top_1pct=bootstrap_differences_v02["1%"],
    difference_top_5pct=bootstrap_differences_v02["5%"],
)

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

# 7. Record experiment provenance and limitations
report = {
    "report_id": "m06_paired_bootstrap_v02",
    "validation_month": 6,
    "validation_records": 108168,
    "validation_fraud_cases": 1450,
    "model_a": "HistGradientBoosting",
    "model_b": "Full-weighted GPU MLP",
    "difference_definition": "MLP minus HGB",
    "method": "paired_transaction_bootstrap",
    "bootstrap_repetitions": 2000,
    "random_seed": 42,
    "interval_method": "95_percentile",
    "fixed_alert_counts": {
        "1%": 1081,
        "5%": 5408,
    },
    "first_500_match_v01": True,
    "results": summary,
    "distribution_file": distribution_path.name,
    "distribution_sha256": sha256_file(distribution_path),
    "input_prediction_manifests": [
        "m06_hgb_reproduction_v01.json",
        "m06_weighted_mlp_predictions_manifest_v01.json",
    ],
    "limitations": [
        "Single validation month",
        "Transaction-level independence assumption",
        "Exploratory selection using the validation data",
        "Not adjusted for multiple model comparisons",
        "Does not establish future production performance",
    ],
    "holdout_status": "SEALED",
}

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

# 8. Display saved results
print("\nEXPANDED BOOTSTRAP ARTIFACT PRESERVATION")
print("=" * 55)

print("Distributions:", distribution_path.name)
print("Report:", report_path.name)

for budget, result in summary.items():
    print(
        f"Top {budget}: "
        f"observed {result['observed_mlp_minus_hgb']:+d}, "
        f"95% CI "
        f"[{result['ci_95_lower']:+.1f}, "
        f"{result['ci_95_upper']:+.1f}], "
        f"MLP favored {result['fraction_favoring_mlp']:.1%}"
    )

print("Month-7 test set: SEALED")
print("Step 06.68: PASS")


### Independently Verify the Expanded Bootstrap Artifacts
Before preparing the Milestone 06 model-selection report, we'll reload the saved results and verify their integrity.

In [ ]:

# ============================================================
# Step 06.69 — Verify Expanded Bootstrap Artifacts
# Read-only | Validation month 6 | Month 7 SEALED
# ============================================================

from pathlib import Path
import hashlib
import json
import numpy as np

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

folder = root / "artifacts" / "milestone_06"

# 1. Load saved report
report_path = folder / "m06_paired_bootstrap_report_v02.json"

report = json.loads(
    report_path.read_text(encoding="utf-8")
)

assert report["validation_month"] == 6
assert report["validation_records"] == 108168
assert report["validation_fraud_cases"] == 1450
assert report["bootstrap_repetitions"] == 2000
assert report["random_seed"] == 42
assert report["holdout_status"] == "SEALED"

print("Experiment provenance: PASS")

# 2. Verify saved distribution checksum
distribution_path = folder / report["distribution_file"]

actual_sha = hashlib.sha256(
    distribution_path.read_bytes()
).hexdigest()

assert actual_sha == report["distribution_sha256"]

print("Expanded bootstrap SHA256: PASS")

# 3. Reload the complete 2,000-resample distributions
with np.load(distribution_path, allow_pickle=False) as saved:
    assert set(saved.files) == {
        "difference_top_1pct",
        "difference_top_5pct",
    }

    distributions = {
        "1%": saved["difference_top_1pct"].copy(),
        "5%": saved["difference_top_5pct"].copy(),
    }

# 4. Confirm the first 500 match the original analysis
with np.load(
    folder / "m06_paired_bootstrap_v01.npz",
    allow_pickle=False,
) as original:

    assert np.array_equal(
        distributions["1%"][:500],
        original["difference_top_1pct"],
    )

    assert np.array_equal(
        distributions["5%"][:500],
        original["difference_top_5pct"],
    )

print("Original 500-resample consistency: PASS")

# 5. Recalculate the results independently
expected_observed = {"1%": 2, "5%": 26}

print("\nEXPANDED BOOTSTRAP INTEGRITY REVIEW")
print("=" * 60)

for budget, differences in distributions.items():

    assert differences.shape == (2000,)
    assert np.issubdtype(differences.dtype, np.integer)

    saved_result = report["results"][budget]

    lower, upper = np.quantile(
        differences, [0.025, 0.975]
    )

    calculations = {
        "ci_95_lower": float(lower),
        "ci_95_upper": float(upper),
        "fraction_favoring_mlp": float(np.mean(differences > 0)),
        "fraction_favoring_hgb": float(np.mean(differences < 0)),
        "fraction_tied": float(np.mean(differences == 0)),
    }

    assert (
        saved_result["observed_mlp_minus_hgb"]
        == expected_observed[budget]
    )

    for metric, value in calculations.items():
        assert np.isclose(
            value,
            saved_result[metric],
            atol=1e-12,
            rtol=0,
        ), f"{budget}: {metric} mismatch"

    print(f"\nTop {budget}: PASS")
    print(f"Observed advantage: {expected_observed[budget]:+d}")
    print(f"95% interval: [{lower:+.1f}, {upper:+.1f}]")
    print(
        "Resamples favoring MLP: "
        f"{calculations['fraction_favoring_mlp']:.1%}"
    )

print("\nSaved 2,000-resample analysis: VERIFIED")
print("Month-7 test set: SEALED")
print("Step 06.69: PASS")


# Milestone 06 — Model Selection and Validation Report

**Project:** Bank Account Fraud Detection — Deep Learning & MLOps  
**Status:** Provisional model selection — validation complete  
**Test dataset:** Month 7 remains sealed

## 1. Objective

Compare traditional machine learning and PyTorch neural networks for detecting fraudulent bank-account applications.

Evaluate overall ranking performance using Precision-Recall AUC (PR-AUC), alongside operational fraud detection at fixed investigation budgets of 1% and 5%.

## 2. Experimental Design

- **Dataset:** Bank Account Fraud Dataset Suite, Base.csv
- **Training:** Months 0–5, 794,989 transactions, 8,151 fraud cases
- **Validation:** Month 6, 108,168 transactions, 1,450 fraud cases
- **Test:** Month 7, sealed
- **Features:** 55 preprocessed features
- **Validation approach:** Chronological holdout
- **Neural-network architecture:** 55 → 64 → 32 → 1
- **Neural-network training:** Adam optimizer, learning rate 0.001, five epochs, batch size 1,024, seed 42

All reported model comparisons use the same validation month and investigation budgets.

## 3. Model Performance

| Model | Validation PR-AUC | Fraud caught, top 1% | Fraud caught, top 5% |
|---|---:|---:|---:|
| HistGradientBoosting | **0.166961** | 275 | 680 |
| Unweighted GPU MLP | 0.160393 | 269 | 690 |
| Weight-25 GPU MLP | 0.161925 | 267 | 701 |
| Full class-weighted GPU MLP | 0.163862 | **277** | **706** |

HistGradientBoosting achieved the highest overall PR-AUC.

The full class-weighted MLP achieved the highest fraud detection at both fixed alert budgets among the evaluated models.

At the 5% budget, the weighted MLP detected 26 additional fraudulent transactions compared with HistGradientBoosting, using the same 5,408 investigations.

## 4. Bootstrap Robustness Analysis

A paired transaction-level bootstrap was conducted using 2,000 resamples, with random seed 42.

| Metric | Top 1% | Top 5% |
|---|---:|---:|
| Observed MLP minus HGB fraud detections | +2 | +26 |
| 95% bootstrap percentile interval | [−22, +22] | [+1, +48] |
| Resamples favoring the MLP | 48.4% | 97.7% |

The 1% comparison is inconclusive because its interval includes zero.

At 5%, the interval remains positive, providing encouraging exploratory evidence of an operational advantage for the weighted MLP.

These are not confirmatory statistical results. The validation period has been used repeatedly for experimentation and model comparison, and the bootstrap does not account for temporal drift, all possible dependence between transactions, or multiple model-selection decisions.

## 5. Calibration and Operational Considerations

Class weighting improved the MLP's fraud-ranking performance but substantially changed its probability outputs.

The fully weighted MLP had a validation Brier score of approximately 0.1282. Its 0.50 classification threshold produced 20,033 false positives and flagged approximately 19.61% of validation transactions.

Therefore, its raw probabilities should not be treated as well-calibrated fraud probabilities.

Fixed investigation budgets offer a more appropriate basis for the current operational comparison. Any future deployment would require further calibration, threshold governance, monitoring, and operational validation.

## 6. Provisional Model Selection

**Deep-learning candidate:** Full class-weighted PyTorch MLP.

**Traditional ML reference:** HistGradientBoosting.

**Primary-metric leader:** HistGradientBoosting, based on validation PR-AUC.

**Operational leader at a 5% investigation budget:** Full class-weighted MLP.

The weighted MLP is provisionally selected as the leading deep-learning model because it detects the most fraud within the predefined 5% investigation capacity.

This does not supersede HistGradientBoosting's stronger overall PR-AUC. Final selection for a deployment scenario depends on which metric represents the actual business objective.

## 7. Limitations and Next Steps

- Preserve all model checkpoints, predictions, manifests, and bootstrap results.
- Investigate calibration and other operational risks before any deployment recommendation.
- Document software dependencies, reproducibility requirements, and decision criteria.
- Complete milestone review and version-control the cleaned notebook and technical documentation.
- Maintain the sealed month-7 test set until the approved final evaluation stage.

**Conclusion:** The full class-weighted MLP is the leading deep-learning candidate based on month-6 operational performance at the 5% investigation budget. HistGradientBoosting remains the stronger model on overall validation PR-AUC. The evidence supports a provisional choice, not a final claim of future production superiority.